# soilhack — feasibility walkthrough (TabPFN-3.5)

This notebook **runs on the GPU workstation** and reproduces the two headline
feasibility results from `RESULTS.md`. No training loop — one `fit()` per task.

Run each cell in order.

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'src'))
print('repo root:', ROOT)


## 0. What model are we actually using?

We **name the checkpoint explicitly** (V3.5). A bare constructor can silently load a
different generation, which would move every number.

In [ ]:
from soilhack import tabpfn_runner as tr
print(tr.describe_version())

## 1. Crop recommendation (classification)

7 soil/climate features -> 22 crops. Watch the accuracy at a **100-row context** —
the few-shot story.

In [ ]:
import numpy as np
from soilhack import data as sdata, metrics as sm
X, y, classes = sdata.load_crop_recommendation()
X = X.to_numpy('float32')
tr_i, te_i = sdata.train_test_split_idx(len(y), 0.7, 42)
rng = np.random.RandomState(42); small = np.sort(rng.choice(tr_i, 100, replace=False))
clf = tr.make_classifier(); clf.fit(X[small], y[small])
proba = tr.predict_class_proba(clf, X[te_i])
print('classes:', len(classes), '| ctx=100 | test=', len(te_i))
print(sm.classification_report(proba, y[te_i], n_classes=len(classes)))

## 2. Soil organic carbon from vis-NIR spectra (regression)

1,051 SNV bands -> SOC. Compare to the constant baseline (the collapse detector).

In [ ]:
from soilhack import data as sdata, metrics as sm
Xs, ys, meta = sdata.load_ossl_soc_sample()
tr_i, te_i = sdata.train_test_split_idx(len(ys), 0.7, 42)
reg = tr.make_regressor(); reg.fit(Xs[tr_i], ys[tr_i])
pred = tr.predict_regression(reg, Xs[te_i])
rep = sm.regression_report(ys[te_i], pred)
const = sm.regression_report(ys[te_i], np.full(len(te_i), ys[tr_i].mean()))
print('TabPFN-3.5:', {k: round(rep[k], 3) for k in ('r2','rmse','rpd','rpiq')})
print('constant  :', {k: round(const[k], 3) for k in ('r2','rmse','rpd')})

## 3. Reading rules

1. A skill number is meaningless without its **context size** and **test set**.
2. Always report the **constant baseline** with a regression number.
3. Classification: accuracy **+ macro-F1 + ECE**, never accuracy alone.